In [2]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sys

In [2]:
path="/sortref/system/rea/glo_bgc/microrys12/microrys3/GLOBAL_MULTIYEAR_BGC_001_033/cmems_mod_glo_bgc_my_0.083deg-lmtl_P1D-i/1998/01/cmems_mod_glo_bgc_my_0.083deg-lmtl_P1D-i_19980101.nc"
ds= xr.open_dataset(path)
ds

<xarray.Dataset> Size: 635MB
Dimensions:       (time: 1, latitude: 2040, longitude: 4320)
Coordinates:
  * time          (time) datetime64[ns] 8B 1998-01-01T12:00:00
  * latitude      (latitude) float32 8kB -80.0 -79.92 -79.83 ... 89.83 89.92
  * longitude     (longitude) float32 17kB -180.0 -179.9 -179.8 ... 179.8 179.9
Data variables:
    zooc          (time, latitude, longitude) float64 71MB ...
    mnkc_epi      (time, latitude, longitude) float64 71MB ...
    mnkc_umeso    (time, latitude, longitude) float64 71MB ...
    mnkc_mumeso   (time, latitude, longitude) float64 71MB ...
    mnkc_lmeso    (time, latitude, longitude) float64 71MB ...
    mnkc_mlmeso   (time, latitude, longitude) float64 71MB ...
    mnkc_hmlmeso  (time, latitude, longitude) float64 71MB ...
    zeu           (time, latitude, longitude) float64 71MB ...
    npp           (time, latitude, longitude) float64 71MB ...
Attributes: (12/14)
    title:                            Global ocean low and mid trophic levels...
    source:                           SEAPODYM-LMTL 3.0.0
    native_grid:                      Yin-Yang overset grid
    references:                       http://www.cls.fr; http://www.seapodym.eu
    institution:                      CLS
    source_physical_variables:        GLOBAL_REANALYSIS_PHY_001_030 CMEMS pro...
    ...                               ...
    date_field:                       1998-01-01
    spatial_resolution:               0.083x0.083
    temporal_resolution:              1 day
    domain:                           global
    history:                          Created on 2024-10-16; Updated micronek...
    Conventions:                      CF-1.8

## Moyenne mensuelle des données journalières

In [17]:
root = Path("/sortref/system/rea/glo_bgc/microrys12/microrys3/GLOBAL_MULTIYEAR_BGC_001_033/cmems_mod_glo_bgc_my_0.083deg-lmtl_P1D-i")
output = Path("/data/rd_exchange/asauvebois/moy_month")
output.mkdir(parents=True, exist_ok=True)

In [ ]:
for year_dir in sorted(root.iterdir()):
    if not year_dir.is_dir():
        continue
    print(f"\nAnnée : {year_dir.name}")
    
    for month_dir in sorted(year_dir.iterdir()):
        if not month_dir.is_dir():
            continue
        print(f"   Mois : {month_dir.name}")
        
        outfile = output/ f"{year_dir.name}_{month_dir.name}_monthly_mean.nc"  # ← ici
        
        if outfile.exists():
            print(f"   → déjà traité, on passe")
            continue
        
        files = sorted(month_dir.glob("*.nc"))
        if len(files) == 0:
            continue
        
        ds = xr.open_mfdataset(
            files,
            combine="by_coords",
            chunks={} 
        )
        monthly_mean = ds.mean(dim="time")
        monthly_mean.to_netcdf(outfile)
        ds.close()


Année : 1998
   Mois : 01
   Mois : 02
   Mois : 03
   Mois : 04
   Mois : 05
   Mois : 06
   Mois : 07
   Mois : 08
   Mois : 09
   Mois : 10
   Mois : 11
   Mois : 12


Petit code pour recalculer les mois manquants

In [14]:


month_dir = Path("/sortref/system/rea/glo_bgc/microrys12/microrys3/GLOBAL_MULTIYEAR_BGC_001_033/cmems_mod_glo_bgc_my_0.083deg-lmtl_P1D-i/2002/07")
outfile = Path("/data/rd_exchange/asauvebois/moy_month/2002_07_monthly_mean.nc")

files = sorted(month_dir.glob("*.nc"))
print(f"{len(files)} fichiers trouvés")

ds = xr.open_mfdataset(files, combine="by_coords", chunks="auto")
monthly_mean = ds.mean(dim="time")
monthly_mean.to_netcdf(outfile)
ds.close()

print("Fichier recréé !")

31 fichiers trouvés
Fichier recréé !


## Vérification des valeurs des moyennes calculées


In [4]:
root2 = Path("/data/rd_exchange/asauvebois/moy_month")

In [7]:
annee=str(2000)
mois= str(11)
lat_cible= 89.83
lon_cible= 179.8
variable= "mnkc_epi"

In [8]:
month_dir = root/annee/mois
outfile = root2 / f"{annee}_{mois}_monthly_mean.nc"

files = sorted(month_dir.glob("*.nc"))

if len(files) == 0:
    print("Aucun fichier trouvé !")
else:
    # Fichiers journaliers
    ds = xr.open_mfdataset(files, combine="by_coords")
    serie = ds[variable].sel(
        latitude=lat_cible,
        longitude=lon_cible,
        method='nearest'
    )
    moy_month_test = serie.mean(dim='time').values

    # Fichier mensuel déjà calculé
    ds2 = xr.open_dataset(outfile)  # open_dataset car c'est un seul fichier
    serie2 = ds2[variable].sel(
        latitude=lat_cible,
        longitude=lon_cible,
        method='nearest'
    )
    moy_month_model = serie2.values

    print(f"Moyenne calculée depuis journaliers : {moy_month_test}")
    print(f"Moyenne depuis fichier mensuel      : {moy_month_model}")

Moyenne calculée depuis journaliers : 15.025775107469947
Moyenne depuis fichier mensuel      : 15.025775107469947


In [3]:
path="/data/rd_exchange/asauvebois/moy_month/1998_10_monthly_mean.nc"
ds= xr.open_dataset(path)
ds

<xarray.Dataset> Size: 635MB
Dimensions:       (latitude: 2040, longitude: 4320)
Coordinates:
  * latitude      (latitude) float32 8kB -80.0 -79.92 -79.83 ... 89.83 89.92
  * longitude     (longitude) float32 17kB -180.0 -179.9 -179.8 ... 179.8 179.9
Data variables:
    zooc          (latitude, longitude) float64 71MB ...
    mnkc_epi      (latitude, longitude) float64 71MB ...
    mnkc_umeso    (latitude, longitude) float64 71MB ...
    mnkc_mumeso   (latitude, longitude) float64 71MB ...
    mnkc_lmeso    (latitude, longitude) float64 71MB ...
    mnkc_mlmeso   (latitude, longitude) float64 71MB ...
    mnkc_hmlmeso  (latitude, longitude) float64 71MB ...
    zeu           (latitude, longitude) float64 71MB ...
    npp           (latitude, longitude) float64 71MB ...
Attributes: (12/14)
    title:                            Global ocean low and mid trophic levels...
    source:                           SEAPODYM-LMTL 3.0.0
    native_grid:                      Yin-Yang overset grid
    references:                       http://www.cls.fr; http://www.seapodym.eu
    institution:                      CLS
    source_physical_variables:        GLOBAL_REANALYSIS_PHY_001_030 CMEMS pro...
    ...                               ...
    date_field:                       1998-10-01
    spatial_resolution:               0.083x0.083
    temporal_resolution:              1 day
    domain:                           global
    history:                          Created on 2024-10-16; Updated micronek...
    Conventions:                      CF-1.8

In [4]:
ds_test = xr.open_dataset("/data/rd_exchange/asauvebois/moy_month/1999_04_monthly_mean.nc")
print(ds_test.data_vars)  # les variables existent ?
for var in ds_test.data_vars:
    print(var, ds_test[var].isnull().sum().values)  # combien de NaN par variable ?

Data variables:
    zooc          (latitude, longitude) float64 71MB ...
    mnkc_epi      (latitude, longitude) float64 71MB ...
    mnkc_umeso    (latitude, longitude) float64 71MB ...
    mnkc_mumeso   (latitude, longitude) float64 71MB ...
    mnkc_lmeso    (latitude, longitude) float64 71MB ...
    mnkc_mlmeso   (latitude, longitude) float64 71MB ...
    mnkc_hmlmeso  (latitude, longitude) float64 71MB ...
    zeu           (latitude, longitude) float64 71MB ...
    npp           (latitude, longitude) float64 71MB ...
zooc 8812800
mnkc_epi 8812800
mnkc_umeso 8812800
mnkc_mumeso 8812800
mnkc_lmeso 8812800
mnkc_mlmeso 8812800
mnkc_hmlmeso 8812800
zeu 8812800
npp 8812800


In [11]:
ds_test = xr.open_dataset("/data/rd_exchange/asauvebois/moy_month/2022_03_monthly_mean.nc")
total = ds_test['mnkc_lmeso'].size
nans = int(ds_test['mnkc_lmeso'].isnull().sum().values)
print(f"Total pixels : {total}")
print(f"NaN : {nans}")
print(f"Pourcentage NaN : {nans/total*100:.1f}%")

Total pixels : 8812800
NaN : 8812800
Pourcentage NaN : 100.0%
